# AI Code Explainer using LangChain

1) Accepts source code,
2) runs static analysis (Python `ast`(**Abstract Syntax Tree**)), and
3) uses an LLM (openAI or Gemini or Cohere) to generate a structured explanation.

- Tested against: `langchain==1.3.6`, `langchain-openai==1.2.2`. 
- User has option to pick up LLM (OpenAI, Gemini, Cohere) 

## 1. Install packages

In [ ]:
# %pip install -q langchain==1.3.6 langchain-openai==1.2.2 pydantic python-dotenv

In [3]:
%pip install langchain-google-genai==4.4.0

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 19.3 MB/s  0:00:00

  Attempting uninstall: google-genai

    Found existing installation: google-genai 1.65.0

    Uninstalling google-genai-1.65.0:

      Successfully uninstalled google-genai-1.65.0

   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ----------------------------------------

In [5]:
%pip install langchain-cohere==0.6.0


   ---------------- ----------------------- 2/5 [fastavro]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ --------------- 3/5 [cohere]
   ------------------------ 

In [ ]:
# check package varsion
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai",
           "pydantic", "langchain-google-genai", "langchain-cohere"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


# langchain version: 1.3.6
# langchain-community version: 0.4.2
# langchain-openai version: 1.2.2
# pydantic version: 2.13.5
# langchain-google-genai version: 4.4.0
# langchain-cohere version: 0.6.0

langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
pydantic version: 2.12.5
langchain-google-genai version: 4.4.0
langchain-cohere version: 0.6.0


## 2. Imports

In [4]:
import os
import ast
import json
from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
# from langchain_cohere import ChatCohere
# from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate

## Choose one of 3 options: OpenAI model / Gemini / Cohere

- You can use openAI(paid) or Gemini(Free) or Cohere(Free)


#### Load environment variables

Create a `.env` file next to the notebook and put any one key

```text
OPENAI_API_KEY=your_api_key_here
GEMINI_API_KEY=your_api_key_here
COHERE_API_KEY=your_api_key_here

### 3.1) OpenAI Set up

In [1]:
import os
import getpass
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
provider="openai"

# 1. LOAD OPENAI API KEY
# Try to load environment variables from a .env file
load_dotenv()

# If the key is not found in the environment,
# prompt the user securely
if not os.getenv("OPENAI_API_KEY"):
    api_key = getpass.getpass(
        "OPENAI_API_KEY not found in environment. "
        "Please enter your API key: "
    )

    if not api_key:
        raise ValueError("OPENAI_API_KEY is required to proceed.")

    os.environ["OPENAI_API_KEY"] = api_key

print("OpenAI API key loaded.")

# 2. TEST OPENAI API CONNECTION USING LANGCHAIN
print("Testing API connection...")

try:
    # Use LangChain to create the test model
    test_llm = ChatOpenAI( model="gpt-4o-mini", temperature=0)

    # Make a minimal API call
    response = test_llm.invoke("Respond with the word 'Success' if you can read this.")

    print(f"API Test: {response.content.strip()}")
    print("Verification complete. OpenAI API key is valid!")

except Exception as e:
    print("\n❌ API Test Failed! "
        "The key might be invalid, expired, or lacking credits."
    )
    print(f"Error details: {e}")


# 3. SELECT MODEL
MODEL_NAME = "gpt-5-mini"

# Other models:
# MODEL_NAME = "gpt-5-nano"
# MODEL_NAME = "gpt-4o-mini"
# MODEL_NAME = "gpt-4o"


# 4. MODEL PARAMETERS
llm_kwargs = {"model": MODEL_NAME}

# GPT-5 and reasoning models may have
# different parameter support.
if not MODEL_NAME.startswith(("gpt-5", "o1", "o3", "o4")):
    llm_kwargs["temperature"] = 0


# 5. INITIALIZE FINAL LANGCHAIN MODEL
llm = ChatOpenAI(**llm_kwargs)
print(f"Using model: {MODEL_NAME}")

KeyboardInterrupt: Interrupted by user

### 3.2) Gemini setup

In [33]:
import os
import getpass
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

provider="gemini"

# 1. LOAD GEMINI API KEY
load_dotenv()


# If the key is not found in the environment,
# prompt the user securely
if not os.getenv("GEMINI_API_KEY"):

    api_key = getpass.getpass(
        "GEMINI_API_KEY not found in environment. "
        "Please enter your API key: "
    )

    if not api_key:
        raise ValueError("GEMINI_API_KEY is required to proceed.")

    os.environ["GEMINI_API_KEY"] = api_key


print("GEMINI_API_KEY loaded.")


# 2. TEST GEMINI API CONNECTION
print("Testing Gemini API connection...")

try:

    # Create a small Gemini model for testing
    test_llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0
    )

    # Make a minimal API call
    response = test_llm.invoke("Respond with the word 'Success' if you can read this.")

    print(f"API Test: {response.content}")

    print(
        "Verification complete. "
        "Your Gemini API key is valid and working!"
    )


except Exception as e:
    print(
        "\n❌ API Test Failed! "
        "The key might be invalid, expired, "
        "or the model may not be available."
    )

    print(f"Error details: {e}")


# 3. SELECT GEMINI MODEL
MODEL_NAME = "gemini-2.5-flash"

# Other Gemini models can be selected here, if available
# MODEL_NAME = "gemini-2.5-flash-lite"
# MODEL_NAME = "gemini-2.5-pro"


# 4. INITIALIZE LANGCHAIN GEMINI MODEL
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0
)

print( f"Using Gemini model: {MODEL_NAME}")

GEMINI_API_KEY loaded.
Testing Gemini API connection...
API Test: Success
Verification complete. Your Gemini API key is valid and working!
Using Gemini model: gemini-2.5-flash


### 3.3) Cohere

In [45]:
import os
import getpass
from dotenv import load_dotenv
from langchain_cohere import ChatCohere

provider = "cohere"


# 1. LOAD COHERE API KEY
load_dotenv()


# If the key is not found in the environment,
# prompt the user securely
if not os.getenv("COHERE_API_KEY"):
    api_key = getpass.getpass(
        "COHERE_API_KEY not found in environment. "
        "Please enter your API key: "
    )

    if not api_key:
        raise ValueError("COHERE_API_KEY is required to proceed.")

    os.environ["COHERE_API_KEY"] = api_key

print("COHERE_API_KEY loaded.")


# 2. TEST COHERE API CONNECTION
print("Testing Cohere API connection...")

try:

    # Create a small Cohere model for testing
    test_llm = ChatCohere(
        model="command-a-03-2025",
        temperature=0
    )

    # Make a minimal API call
    response = test_llm.invoke("Respond with the word 'Success' if you can read this.")

    print(f"API Test: {response.content}")

    print(
        "Verification complete. "
        "Your Cohere API key is valid and working!"
    )


except Exception as e:

    print(
        "\n❌ API Test Failed! "
        "The key might be invalid, expired, "
        "or the model may not be available."
    )

    print(f"Error details: {e}")


# 3. SELECT COHERE MODEL
MODEL_NAME = "command-a-03-2025"

# Other Cohere models can be selected here,
# depending on availability in your Cohere account.
#
# MODEL_NAME = "command-r-plus"
# MODEL_NAME = "command-r"


# 4. INITIALIZE LANGCHAIN COHERE MODEL
llm = ChatCohere(model=MODEL_NAME, temperature=0)

print(f"Using Cohere model: {MODEL_NAME}")


COHERE_API_KEY loaded.
Testing Cohere API connection...
API Test: Success
Verification complete. Your Cohere API key is valid and working!
Using Cohere model: command-a-03-2025


## 4. Structured output schema

In [46]:
class FunctionInfo(BaseModel):
    name:      str = Field(description="Name of the function")
    arguments: List[str] = Field(description="Arguments accepted by the function")
    purpose:   str = Field(description="What the function does")


class CodeExplanation(BaseModel):
    language:             str = Field(description="Programming language of the code")
    overview:             str = Field(description="Short explanation of what the complete program does")
    inputs:               List[str] = Field(description="Inputs used by the program")
    outputs:              List[str] = Field(description="Outputs produced by the program")
    functions:            List[FunctionInfo] = Field(description="Functions present in the program")
    execution_flow:       List[str] = Field(description="Step-by-step execution flow")
    important_concepts:   List[str] = Field(description="Important programming concepts demonstrated")
    detailed_explanation: str = Field(description="Detailed beginner-friendly explanation")
    possible_errors:      List[str] = Field(description="Potential errors or edge cases visible from the code")

## 5. Structured LLM

In [47]:
# The structured LLM is little different for openai, gemini
if provider =="openai":
    structured_llm = llm.with_structured_output(CodeExplanation, method="function_calling",)
    
elif provider =="gemini":
    structured_llm = llm.with_structured_output(CodeExplanation, method="json_schema")
    
elif provider == "cohere":
    structured_llm = llm.with_structured_output(CodeExplanation, method="json_schema")

else:
    raise ValueError(f"Unsupported provider: {provider}")

print("Using provide: ", provider)

Using provide:  cohere


## 6. Python AST analyzer

Extracts functions, classes, imports, variables, calls, and syntax errors.

In [48]:
def analyze_python_code(code: str) -> dict:
    try:
        tree = ast.parse(code)
        
    except SyntaxError as e:
        return {
            "valid_python": False,
            "syntax_error": str(e),
            "functions": [],
            "classes": [],
            "imports": [],
            "variables": [],
            "function_calls": [],
        }

    functions, classes = [], []
    imports, variables, function_calls = set(), set(), set()

    for node in ast.walk(tree):

        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            a = node.args
            arguments = [x.arg for x in a.posonlyargs + a.args + a.kwonlyargs]
            if a.vararg:
                arguments.append("*" + a.vararg.arg)
            if a.kwarg:
                arguments.append("**" + a.kwarg.arg)
            functions.append({
                "name": node.name,
                "arguments": arguments,
                "line": node.lineno,
            })

        elif isinstance(node, ast.ClassDef):
            classes.append({"name": node.name, "line": node.lineno})

        elif isinstance(node, ast.Import):
            for alias in node.names:
                imports.add(alias.name)

        elif isinstance(node, ast.ImportFrom):
            module = node.module or ""
            for alias in node.names:
                imports.add(f"{module}.{alias.name}")

        elif isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    variables.add(target.id)

        elif isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                function_calls.add(node.func.id)
            elif isinstance(node.func, ast.Attribute):
                function_calls.add(node.func.attr)

    return {
        "valid_python": True,
        "syntax_error": None,
        "functions": sorted(functions, key=lambda f: f["line"]),
        "classes": sorted(classes, key=lambda c: c["line"]),
        "imports": sorted(imports),
        "variables": sorted(variables),
        "function_calls": sorted(function_calls),
    }

## 7. Test the analyzer

In [ ]:
# Following code would go inside the application to be explained by the LLM.

def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)


numbers = [10, 20, 30, 40, 50]

average = calculate_average(numbers)

print("Average:", average)


In [49]:
code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)


numbers = [10, 20, 30, 40, 50]

average = calculate_average(numbers)

print("Average:", average)
"""

analysis = analyze_python_code(code)
print(json.dumps(analysis, indent=4))

{
    "valid_python": true,
    "syntax_error": null,
    "functions": [
        {
            "name": "calculate_average",
            "arguments": [
                "numbers"
            ],
            "line": 2
        }
    ],
    "classes": [],
    "imports": [],
    "variables": [
        "average",
        "numbers",
        "total"
    ],
    "function_calls": [
        "calculate_average",
        "len",
        "print",
        "sum"
    ]
}


## 9. Prompt

In [50]:
system_prompt = """
    You are an expert programming instructor.

    Your task is to explain source code accurately.
    
    The user will provide:
    1. Source code
    2. Programming language
    3. Static analysis information when available
    
    Rules:
    - Explain what the code actually does.
    - Do not invent functionality thaw t is not present.
    - Follothe actual execution flow.
    - Explain functions and classes that are present.
    - Identify inputs and outputs.
    - Explain important programming concepts.
    - Mention potential errors or edge cases that can reasonably be identified.
    - Keep the explanation understandable to a beginner.
    - Preserve technical accuracy.
    - If static analysis conflicts with the source code, use the source code as the authority.
"""

human_prompt = """
    Programming Language:
    {language}
    
    Static Code Analysis:
    {analysis}
    
    Source Code:
    ```text
    {code}
    ```
    
    Generate a complete explanation of this code.
"""

prompt = ChatPromptTemplate.from_messages([
    ( "system", system_prompt,),
    ( "human",human_prompt,),
])

## 10. Chain

In [51]:
chain = prompt | structured_llm

## 11. Explanation function

In [52]:
def explain_code(code: str, language: str, analysis: dict) -> CodeExplanation:
    return chain.invoke({
        "language": language,
        "analysis": json.dumps(analysis, indent=2),
        "code": code,
    })

## 12. Run the pipeline

In [53]:
language = "Python"
analysis = analyze_python_code(code)

explanation = explain_code(code=code, language=language, analysis=analysis)

## 13. Inspect the raw structured result

In [54]:
explanation

CodeExplanation(language='Python', overview='This program calculates the average of a list of numbers and prints the result.', inputs=['A list of numbers (numbers)'], outputs=['The average of the numbers (printed to the console)'], functions=[FunctionInfo(name='calculate_average', arguments=['numbers'], purpose='Calculates the average of a list of numbers by summing them and dividing by the count of numbers.')], execution_flow=['Define the function `calculate_average` that takes a list of numbers as input.', 'Inside the function, calculate the sum of the numbers using the `sum` function.', 'Divide the total sum by the number of elements in the list using the `len` function to compute the average.', 'Return the calculated average.', 'Create a list of numbers `[10, 20, 30, 40, 50]`.', 'Call the `calculate_average` function with the list of numbers and store the result in the variable `average`.', 'Print the calculated average to the console.'], important_concepts=['Function definition an

In [55]:
print(explanation.model_dump_json(indent=4))

{
    "language": "Python",
    "overview": "This program calculates the average of a list of numbers and prints the result.",
    "inputs": [
        "A list of numbers (numbers)"
    ],
    "outputs": [
        "The average of the numbers (printed to the console)"
    ],
    "functions": [
        {
            "name": "calculate_average",
            "arguments": [
                "numbers"
            ],
            "purpose": "Calculates the average of a list of numbers by summing them and dividing by the count of numbers."
        }
    ],
    "execution_flow": [
        "Define the function `calculate_average` that takes a list of numbers as input.",
        "Inside the function, calculate the sum of the numbers using the `sum` function.",
        "Divide the total sum by the number of elements in the list using the `len` function to compute the average.",
        "Return the calculated average.",
        "Create a list of numbers `[10, 20, 30, 40, 50]`.",
        "Call the `cal

## 14. Display helper

In [56]:
def display_explanation(exp: CodeExplanation) -> None:
    def header(title):
        print(f"\n{title}")
        print("=" * 60)

    header("OVERVIEW")
    print(exp.overview)

    header("INPUTS")
    for item in exp.inputs:
        print(f"- {item}")

    header("OUTPUTS")
    for item in exp.outputs:
        print(f"- {item}")

    header("FUNCTIONS")
    for fn in exp.functions:
        args = ", ".join(fn.arguments) if fn.arguments else "None"
        print(f"\nFunction: {fn.name}")
        print(f"Arguments: {args}")
        print(f"Purpose: {fn.purpose}")

    header("EXECUTION FLOW")
    for i, step in enumerate(exp.execution_flow, start=1):
        print(f"{i}. {step}")

    header("IMPORTANT PROGRAMMING CONCEPTS")
    for concept in exp.important_concepts:
        print(f"- {concept}")

    header("DETAILED EXPLANATION")
    print(exp.detailed_explanation)

    header("POSSIBLE ERRORS / EDGE CASES")
    if exp.possible_errors:
        for err in exp.possible_errors:
            print(f"- {err}")
    else:
        print("No obvious errors or edge cases identified.")

In [57]:
display_explanation(explanation)


OVERVIEW
This program calculates the average of a list of numbers and prints the result.

INPUTS
- A list of numbers (numbers)

OUTPUTS
- The average of the numbers (printed to the console)

FUNCTIONS

Function: calculate_average
Arguments: numbers
Purpose: Calculates the average of a list of numbers by summing them and dividing by the count of numbers.

EXECUTION FLOW
1. Define the function `calculate_average` that takes a list of numbers as input.
2. Inside the function, calculate the sum of the numbers using the `sum` function.
3. Divide the total sum by the number of elements in the list using the `len` function to compute the average.
4. Return the calculated average.
5. Create a list of numbers `[10, 20, 30, 40, 50]`.
6. Call the `calculate_average` function with the list of numbers and store the result in the variable `average`.
7. Print the calculated average to the console.

IMPORTANT PROGRAMMING CONCEPTS
- Function definition and invocation
- List manipulation
- Built-in fun

# STOP

## Following is optional

## 15. One function to run everything

In [21]:
def generate_code_explanation(code: str, language: str = "Python") -> CodeExplanation:
    if language.lower() == "python":
        analysis = analyze_python_code(code)
    else:
        analysis = {
            "static_analysis_available": False,
            "message": "AST analysis is only implemented for Python.",
        }

    return explain_code(code=code, language=language, analysis=analysis)

## 16. Test with another program

In [22]:
code2 = """
def factorial(n):

    if n == 0:
        return 1

    return n * factorial(n - 1)


number = 5

result = factorial(number)

print(result)
"""

explanation2 = generate_code_explanation(code=code2, language="Python")
display_explanation(explanation2)


OVERVIEW
Defines a recursive function factorial(n) that computes the factorial of a non-negative integer n using recursion, then calls it with number = 5 and prints the result (120).

INPUTS
- n (argument to factorial)
- number = 5 (the concrete input passed to factorial)

OUTPUTS
- Prints the integer 120 to standard output

FUNCTIONS

Function: factorial
Arguments: n
Purpose: Return n! (factorial of n). If n == 0 returns 1 (base case); otherwise returns n * factorial(n - 1) (recursive case).

EXECUTION FLOW
1. The Python interpreter reads and defines the function factorial(n).
2. Execution reaches the assignment number = 5, creating a variable number with value 5.
3. The code calls factorial(number), which starts the recursive computation.
4. Recursive calls: factorial(5) -> 5 * factorial(4) -> 5 * (4 * factorial(3)) -> ... -> 5 * 4 * 3 * 2 * 1 * factorial(0).
5. When factorial(0) is reached, the base case returns 1, and the recursive calls return back up multiplying the values to pr

## 17. Your own code

In [23]:
new_code = """
def is_even(number):
    return number % 2 == 0


numbers = [1, 2, 3, 4, 5, 6]

for number in numbers:
    if is_even(number):
        print(number)
"""

result = generate_code_explanation(new_code, language="Python")
display_explanation(result)


OVERVIEW
This small Python script defines a helper function is_even(number) that checks whether a given integer is even, creates a list of integers, and then iterates through the list printing each even number to the console.

INPUTS
- numbers list: [1, 2, 3, 4, 5, 6] (internal)
- function parameter: number (each element from the numbers list when iterating)

OUTPUTS
- Printed lines to standard output: 2, 4, 6 (each on its own line)

FUNCTIONS

Function: is_even
Arguments: number
Purpose: Returns True if the provided number is even (i.e., divisible by 2), otherwise returns False.

EXECUTION FLOW
1. The interpreter defines the function is_even(number).
2. The list numbers = [1, 2, 3, 4, 5, 6] is created and assigned to the variable numbers.
3. A for loop begins that iterates over each element in numbers, binding each element in turn to the variable number.
4. For each number, the code calls is_even(number).
5. If is_even(number) returns True, the code executes print(number), which writ

# STOP

# Further enhancement

**Easy**
- Support more languages by adding JS/Java/C++ analyzers (regex-based, or use `esprima`/`tree-sitter` for a real parser) alongside the Python `ast` one.
- Syntax highlighting in output — pipe the source through `pygments` before printing.
- Export explanations to Markdown/PDF/HTML instead of just printing to console.
- Batch mode — point it at a folder of `.py` files and generate one explanation per file.

**Medium**
- Complexity metrics — cyclomatic complexity, line count, nesting depth, computed from the AST and fed into the prompt so the LLM's explanation reflects actual complexity.
- Docstring/type-hint checker — flag functions missing docstrings or type hints as part of `possible_errors`.
- Diff-aware explanations — given two versions of a file, explain *what changed* and why, using `difflib` plus the AST comparison.
- Caching — hash the source code, skip the LLM call if it's been explained before (useful for CI or large codebases).
- Interactive mode — wrap it in a simple Gradio/Streamlit UI so non-technical users can paste code and get explanations without touching the notebook.

**Harder**
- Call graph / dependency visualization — use the `function_calls` data already extracted to build a graph (`networkx` + `matplotlib` or `graphviz`) showing which functions call which.
- Security/lint integration — run `bandit` or `pylint` on the code first, feed findings into the prompt so the LLM explains *and* critiques the code.
- Test generation — extend `CodeExplanation` with a `suggested_tests` field, have the LLM propose `pytest` cases based on the execution flow it identified.
- RAG over a codebase — instead of one file, chunk and embed an entire repo (this is where `langchain-community`, which you had installed but the notebook didn't use, would actually earn its place) so students can ask "explain how `X` is used across the project."
- Multi-provider support — apply the same `init_chat_model` pattern from your other two notebooks so it isn't locked to OpenAI.
